# Basic data structures

In [ ]:
# RUN ME FOR EMBEDING VIDEOS AND URLS
from IPython.display import HTML
from urllib.parse import urlparse, parse_qs

def youtube_embed_url(url):
    """Convert any YouTube URL to embed format."""
    parsed = urlparse(url)
    if parsed.netloc == "youtu.be":
        vid = parsed.path.lstrip("/")
    elif "shorts" in parsed.path:
        vid = parsed.path.split("/shorts/")[-1]
    elif "embed" in parsed.path:
        return url  # already embed format, do nothing
    else:
        vid = parse_qs(parsed.query).get("v", [None])[0]
    return f"https://www.youtube.com/embed/{vid}" if vid else url

def embed(url, caption="", width="90%", aspect=(16, 9)):
    if "youtube.com" in url or "youtu.be" in url:
        url = youtube_embed_url(url)
    w, h = aspect
    padding = float(width.strip("%")) * h / w
    return HTML(f'''
<div style="display:flex; flex-direction:column; align-items:center; width:100%;">
  <div style="position:relative; width:{width}; padding-bottom:{padding:.2f}%; height:0; overflow:hidden;">
    <iframe src="{url}"
      style="position:absolute; top:0; left:0; width:100%; height:100%; border:0;"
      allowfullscreen title="{caption}">
    </iframe>
  </div>
  {"<p style='font-style:italic; margin-top:0.4em;'>" + caption + "</p>" if caption else ""}
</div>''')

import base64

def embed_html_file(path, caption="", width="100%", height="800px"):
    """Embed a local standalone HTML file directly into the notebook output
    (no external file dependency at build/render time)."""
    with open(path, "r", encoding="utf-8") as f:
        html_content = f.read()
    b64 = base64.b64encode(html_content.encode("utf-8")).decode("ascii")
    src = f"data:text/html;base64,{b64}"
    return HTML(f'''
<div style="display:flex; flex-direction:column; align-items:center; width:100%;">
  <iframe src="{src}"
    style="width:{width}; height:{height}; border:0;"
    allowfullscreen title="{caption}">
  </iframe>
  {"<p style='font-style:italic; margin-top:0.4em;'>" + caption + "</p>" if caption else ""}
</div>''')



## Lists

Lists are heterogeneous and dynamic arrays which allows to group data which are not necessarily of the same type. Lists are very easy to use but, be careful, they are not the fastest struct you can use for numerical computations.

### Performance

In [ ]:
embed("https://benjdd.com/languages2/", aspect=(16, 16))

For more info about caching, see [https://planetscale.com/blog/caching/](https://planetscale.com/blog/caching/)

In [ ]:
embed_html_file("assets/latency_comparison.html")

In [ ]:
embed_html_file("assets/list_vs_array.html", height="1000px")

### Usage
A list is declared using the ```[ ]``` characters, and its elements are separated by ```,```.

**NOTE**: Lists are NOT recommended for numerical work, it is better to use numpy arrays. 

See: https://quickref.me/python#python-lists

In [ ]:
xdata = [] # declares empty list
print (xdata)
xdata = [1,2, 3, 6.5, 'hello'] # heterogeneous list
print (xdata)

In [ ]:
# Access by indices
print (xdata[0])
print (xdata[3])
print (xdata[4])
print (xdata[4][3]) # Does this make sense?

### Slicing

In [ ]:
embed_html_file("assets/slice_explorer.html")

In [ ]:
# Slicing operations [start=0, end, increment]
print (xdata[0:2])
print (xdata[0:3])
print (xdata[0:4:1])
print (xdata[:4:2])
print (xdata[::1])
print (xdata[::-1]) # inverse order
print (xdata[:-1])

### Aliasing vs. copying

A list variable does not hold the data itself — it holds a *reference* to it. Assigning
`b = a` gives you a second name for the *same* list; it does not create a new one. This
matters a lot once you start passing lists of positions, velocities, etc. into functions.

```python
a = [1, 2, 3]
b = a          # b is just another name for a's list
b.append(99)
print(a)       # a changed too!
```

To get an independent list, use `.copy()` (a *shallow* copy) or, if the list contains
other mutable objects (e.g. a list of lists), `copy.deepcopy()`:

```python
import copy
a = [1, 2, 3]
b = a.copy()          # independent top-level copy
b.append(99)
print(a, b)            # a is untouched

nested = [[1, 2], [3, 4]]
shallow = nested.copy()
shallow[0].append(999)
print(nested)           # inner list is SHARED — this still changes nested!

deep = copy.deepcopy(nested)
deep[0].append(-1)
print(nested)            # untouched this time
```

**Rule of thumb:** if a function receives a list and modifies it in place, that change is
visible to the caller. If you don't want that, copy first — or better, have the function
return a new list instead of mutating its argument.

### Other topics

In [ ]:
# From other lists
a = [2,3,4]
b = [3,4,5]
c = a + b
print (c) # Concatenate the lists
print (2*c) # duplicate the list

In [ ]:
# Check memory address
a.append(55)
print(f"{id(a[0])}, {id(a[1])}, {id(a[2])}, {id(a[3])}")

In [ ]:
import numpy as np
a = np.array([2,3,4])
b = np.array([3,4,5])
c = a + b
print (c)
print(2*c)

In [ ]:
# List comprehension
squares = [x**2 for x in range(0, 10)]
print(squares)

### Looping with `enumerate()` and `zip()`

Two patterns you'll use constantly once you start writing simulations: `enumerate()` gives
you the index *and* the value together, and `zip()` lets you walk through several lists in
lockstep — e.g. a particle's position and velocity at the same timestep.

In [ ]:
positions  = [0.0, 1.2, 2.5, 3.1]
velocities = [0.5, 0.4, -0.2, 0.1]

# enumerate: index + value, no manual counter needed
for i, x in enumerate(positions):
    print(f"particle {i}: x = {x}")

In [ ]:
# zip: walk two (or more) lists together
for x, v in zip(positions, velocities):
    print(f"x = {x:5.2f}, v = {v:5.2f}")

In [ ]:
# combine both: update each position by one Euler step, keeping track of which particle it is
dt = 0.1
new_positions = []
for i, (x, v) in enumerate(zip(positions, velocities)):
    x_new = x + v * dt
    new_positions.append(x_new)
    print(f"particle {i}: {x:5.2f} -> {x_new:5.2f}")

print(new_positions)

### Exercises

*Aliasing bug hunt*

The function below is supposed to return a **new** list
with every element doubled, leaving the original untouched. Predict what `original` and
`result` will print **before** running the cell — then run it, see if you were right, and
fix the function so it no longer modifies its argument.

In [ ]:
def double_in_place(data):
    for i in range(len(data)):
        data[i] = data[i] * 2
    return data

original = [1, 2, 3]
result = double_in_place(original)
print("original:", original)
print("result:  ", result)

# YOUR FIX HERE: rewrite double_in_place so `original` is left unchanged

**Exercise:** *Debug the AI's code.* The snippet below claims to return the prime factors
of `n`, and was pasted in from an AI assistant without checking it carefully. Find the bug,
explain in one sentence what goes wrong and why it still often "looks like it works," and
fix it.

In [ ]:
def primefactors_buggy(n):
    """
    From an AI assistant — has not been checked.
    """
    factors = []
    divisor = 2
    while divisor <= n:
        if n % divisor == 0:
            factors.append(divisor)
            n /= divisor
        else:
            divisor += 1
    return factors

print(primefactors_buggy(360))
print(type(360 / 2))  # look at this closely — what type is `n` after the first division?

# YOUR FIX HERE
def primefactors(n):
    # YOUR CODE HERE
    pass

**Exercise:** *Prove the claim.* The lecture states lists are "not the fastest struct... for
numerical computations." Prove it. Build a large list `x = list(range(N))` and an equivalent
`numpy` array `a = np.array(x)` for `N = 1_000_000`, then use `%timeit` to compare:

1. summing `x` with an explicit `for` loop and a running total,
2. summing `x` with the builtin `sum(x)`,
3. summing `a` with `a.sum()`.

Report the three timings and, by trying a few different values of `N`, estimate roughly
where `a.sum()` stops being dominated by fixed overhead and starts actually winning on the
arithmetic itself.

In [ ]:
import numpy as np

N = 1_000_000
x = list(range(N))
a = np.array(x)

# YOUR CODE HERE — use %timeit on each of the three approaches

<div class="alert alert-block alert-info">
<b>Exercise:</b> Write a function that receives a list and prints it in reverse order every 2 elements
</div>

In [ ]:
def printreversed(l):
    # YOUR CODE HERE
    pass

In [ ]:
printreversed([1, 2, 3])
printreversed([1, 2, 3, 4 , "hola", "mundo"])

<div class="alert alert-block alert-info">
<b>Exercise:</b> Write a function that receives a positive integer and returns a list with all its prime factors
</div>

In [ ]:
import numpy as np

def primefactors(n):
    # YOUR CODE HERE
    pass

# def primefactors(n):
#     """
#     From chatgpt
#     """
#     factors = []
#     divisor = 2
#     while divisor <= n:
#         if n % divisor == 0:
#             factors.append(divisor)
#             n //= divisor
#         else:
#             divisor += 1
#     return factors
    
def isprime(n):
    # YOUR CODE HERE
    pass

In [ ]:
print(primefactors(2))
print(primefactors(8))
print(primefactors(10))
print(primefactors(201))
print(primefactors(97))
print(primefactors(1213427))
print(primefactors(1213428))
#print(primefactors(1213428987543096)) # Slow


<div class="alert alert-block alert-info">
<b>Exercise:</b> Write a function that receives a list of numbers and returns its mean, max and min values . Use numpy
</div>

In [ ]:
import numpy as np

def stats(data):
# YOUR CODE HERE
pass

In [ ]:
stats([1, 2, 3.5])

## Tuples
A tuple is like a list, but is inmutable, it cannot change. It is declared by using ```()```.

In [ ]:
a = (1, 2)
print (a)
print (a[0])
# a[1] = 4 # error, tuple is inmutable
b = () # empty tuple
print (b)

You can use tuples to unpack data from functions returning several results

In [ ]:
def func(x, y) :
    return x + y, x-y # returns a tuple

a, b = func(1, 2)
print (a, b)

## Classes
Python is an object oriented language. Everything is an object. You can also create new types by using classes, after defining their attributes and methods. When creating classes, you should is the ```self``` keyword, which is the analogous to the pointer ```this``` in c++. Let's create a class for a point.

In [ ]:
class Point2D : 
    """This is a doctring. This allows to embed documentation inside the class definition.
    You can split it 
    across several lines.
    """
    def __init__(self, x = 0, y = 0): 
        """ This is the constructor"""
        self.x_ = x # attribute x_
        self.y_ = y # attribute y_
        
    def coordinates(self):
        return self.x_, self.y_
    
    def __str__(self):
        """Cast method to convert to string"""
        return (f"Coordinates : ( {self.x_:25.16e}, {self.y_:25.16e} )")

In [ ]:
p1 = Point2D() # constructs a point with default internal attributes
print (p1) # Uses the str cast method

p2 = Point2D(2, -3)
print (p2)

In [ ]:
embed("https://pythontutor.com/visualize.html#code=class%20Point2D%20%3A%0A%20%20%20%20%22%22%22This%20is%20a%20doctring.%20This%20allows%20to%20embed%20documentation%20inside%20the%20class%20definition.%0A%20%20%20%20You%20can%20split%20it%20%0A%20%20%20%20across%20several%20lines.%0A%20%20%20%20%22%22%22%0A%20%20%20%20def%20__init__%28self,%20x%20%3D%200,%20y%20%3D%200%29%3A%20%0A%20%20%20%20%20%20%20%20%22%22%22%20This%20is%20the%20constructor%22%22%22%0A%20%20%20%20%20%20%20%20self.x_%20%3D%20x%20%23%20attribute%20x_%0A%20%20%20%20%20%20%20%20self.y_%20%3D%20y%20%23%20attribute%20y_%0A%20%20%20%20%20%20%20%20%0A%20%20%20%20def%20coordinates%28self%29%3A%0A%20%20%20%20%20%20%20%20return%20self.x_,%20self.y_%0A%20%20%20%20%0A%20%20%20%20def%20__str__%28self%29%3A%0A%20%20%20%20%20%20%20%20%22%22%22Cast%20method%20to%20convert%20to%20string%22%22%22%0A%20%20%20%20%20%20%20%20return%20%28f%22Coordinates%20%3A%20%28%20%7Bself.x_%3A25.16e%7D,%20%7Bself.y_%3A25.16e%7D%20%29%22%29%0A%20%20%20%20%20%20%20%20%0A%0Ap1%20%3D%20Point2D%28%29%20%23%20constructs%20a%20point%20with%20default%20internal%20attributes%0Aprint%20%28p1%29%20%23%20Uses%20the%20str%20cast%20method%0A%0Ap2%20%3D%20Point2D%282,%20-3%29%0Aprint%20%28p2%29&curInstr=0&mode=display&origin=opt-frontend.js&py=3")

You can save the class to a file, and then later import it for re-use (you can import any python code)

In [ ]:
%%html
<iframe width="800" height="500" frameborder="0" src="https://pythontutor.com/iframe-embed.html#code=class%20Point2D%20%3A%0A%20%20%20%20%22%22%22This%20is%20a%20doctring.%20This%20allows%20to%20embed%20documentation%20inside%20the%20class%20definition.%0A%20%20%20%20You%20can%20split%20it%20%0A%20%20%20%20across%20several%20lines.%0A%20%20%20%20%22%22%22%0A%20%20%20%20def%20__init__%28self,%20x%20%3D%200,%20y%20%3D%200%29%3A%20%0A%20%20%20%20%20%20%20%20%22%22%22%20This%20is%20the%20constructor%22%22%22%0A%20%20%20%20%20%20%20%20self.x_%20%3D%20x%20%23%20attribute%20x_%0A%20%20%20%20%20%20%20%20self.y_%20%3D%20y%20%23%20attribute%20y_%0A%20%20%20%20%20%20%20%20%0A%20%20%20%20def%20coordinates%28self%29%3A%0A%20%20%20%20%20%20%20%20return%20self.x_,%20self.y_%0A%20%20%20%20%0A%20%20%20%20def%20__str__%28self%29%3A%0A%20%20%20%20%20%20%20%20%22%22%22Cast%20method%20to%20convert%20to%20string%22%22%22%0A%20%20%20%20%20%20%20%20return%20%28f%22Coordinates%20%3A%20%28%20%7Bself.x_%3A25.16e%7D,%20%7Bself.y_%3A25.16e%7D%20%29%22%29%0A%20%20%20%20%20%20%20%20%0A%0Ap1%20%3D%20Point2D%28%29%20%23%20constructs%20a%20point%20with%20default%20internal%20attributes%0Aprint%20%28p1%29%20%23%20Uses%20the%20str%20cast%20method%0A%0Ap2%20%3D%20Point2D%282,%20-3%29%0Aprint%20%28p2%29&codeDivHeight=400&codeDivWidth=350&curInstr=0&origin=opt-frontend.js&py=3"> </iframe>


In [ ]:
%%file Point3D.py 
class Point3D :
    """This is a doctring. This allows to embed documentation inside the class definition.
    You can split it 
    across several lines.
    """
    def __init__(self, x = 0, y = 0, z = 0): 
        """ This is the constructor"""
        self.x_ = x # attribute x_
        self.y_ = y # attribute y_
        self.z_ = z # attribute z_
        
    def coordinates(self):
        return self.x_, self.y_, self.z_
    
    def __str__(self):
        """Cast method to convert to string"""
        return (f"Coordinates : ( {self.x_:25.16e}, {self.y_:25.16e}, , {self.z_:25.16e} )")

In [ ]:
import Point3D as P3D
p3 = P3D.Point3D(2, 5, 0.9)
print (p3)

## Dictionaries

Dictionaries are the analogous of associative memories or associative arrays. Basically, they are a generalized container where the key is not necessarily an integer but an arbitrary object of inmutable type, called a key (for example, tuples, a reange of number, etc, nut not a list of integers, since the last is mutable). 

Dictionaries can be seen as unordered sets of the pairs _key:value_, and are sourrounded by curly braces `{}`. It is posible to delete/acces/add/etc values by using the corresponding key or key/value pair. The `keys()` method for a dictionary returns the keys of that dictionary. To check for a given key, you can use the keyword `in`.

Refs:

- https://realpython.com/python-dicts/
- https://quickref.me/python#python-data-types

In [ ]:
# creates a dictionary with several key:value pairs. Keys are strings
tel = {'jack': 4098, 'sape': 4139}  
print (tel)
# acces  by key. If key does not exists, creates a new entry
tel['guido'] = 4127 
print (tel)
# Access by key
print (tel['jack'])
# Delete by key
del tel['sape']    
print (tel)
# key a list of the keys
print(tel.keys())   
# list of values
print(tel.values())
# keys and values
print(tel.items())

In [ ]:
%%html
<iframe width="800" height="500" frameborder="0" src="https://pythontutor.com/iframe-embed.html#code=%23%20creates%20a%20dictionary%20with%20several%20key%3Avalue%20pairs.%20Keys%20are%20strings%0Atel%20%3D%20%7B'jack'%3A%204098,%20'sape'%3A%204139%7D%20%20%0Aprint%20%28tel%29%0A%23%20acces%20%20by%20key.%20If%20key%20does%20not%20exists,%20creates%20a%20new%20entry%0Atel%5B'guido'%5D%20%3D%204127%20%0Aprint%20%28tel%29%0A%23%20Access%20by%20key%0Aprint%20%28tel%5B'jack'%5D%29%0A%23%20Delete%20by%20key%0Adel%20tel%5B'sape'%5D%20%20%20%20%0Aprint%20%28tel%29%0A%23%20key%20a%20list%20of%20the%20keys%0Aprint%28tel.keys%28%29%29%20%20%20%0A%23%20list%20of%20values%0Aprint%28tel.values%28%29%29%0A%23%20keys%20and%20values%0Aprint%28tel.items%28%29%29&codeDivHeight=400&codeDivWidth=350&cumulative=false&curInstr=0&heapPrimitives=nevernest&origin=opt-frontend.js&py=3&rawInputLstJSON=%5B%5D&textReferences=false"> </iframe>

In [ ]:
embed("https://pythontutor.com/visualize.html#code=%23%20creates%20a%20dictionary%20with%20several%20key%3Avalue%20pairs.%20Keys%20are%20strings%0Atel%20%3D%20%7B'jack'%3A%204098,%20'sape'%3A%204139%7D%20%20%0Aprint%20%28tel%29%0A%23%20acces%20%20by%20key.%20If%20key%20does%20not%20exists,%20creates%20a%20new%20entry%0Atel%5B'guido'%5D%20%3D%204127%20%0Aprint%20%28tel%29%0A%23%20Access%20by%20key%0Aprint%20%28tel%5B'jack'%5D%29%0A%23%20Delete%20by%20key%0Adel%20tel%5B'sape'%5D%20%20%20%20%0Aprint%20%28tel%29%0A%23%20key%20a%20list%20of%20the%20keys%0Aprint%28tel.keys%28%29%29%20%20%20%0A%23%20list%20of%20values%0Aprint%28tel.values%28%29%29%0A%23%20keys%20and%20values%0Aprint%28tel.items%28%29%29&curInstr=0&mode=display&origin=opt-frontend.js&py=3")

In [ ]:
for key, val in tel.items():
    print(f"{key=}, {val=}")

In [ ]:
'guido' in tel   # check if a key is in the dictionary

### Dictionary comprehensions

Just like list comprehensions build a list in one line, dictionary comprehensions build a
dictionary in one line: `{key_expr: value_expr for item in iterable}`.

In [ ]:
# squares of 0..9, keyed by the number itself
squares = {n: n**2 for n in range(10)}
print(squares)

In [ ]:
# transform an existing dictionary: convert values, keep the keys
tel = {'jack': 4098, 'sape': 4139, 'guido': 4127}
tel_str = {name: f"+57-{number}" for name, number in tel.items()}
print(tel_str)

In [ ]:
# with a filter condition
tel_area_41 = {name: number for name, number in tel.items() if str(number).startswith('41')}
print(tel_area_41)

## Exercises 
Some based on  "A Primer on Scientific Programming with Python", Lantangen, Springer

### Aliasing bug hunt
The function below is supposed to return a **new** list
with every element doubled, leaving the original untouched. Predict what `original` and
`result` will print **before** running the cell — then run it, see if you were right, and
fix the function so it no longer modifies its argument.

In [ ]:
def double_in_place(data):
    for i in range(len(data)):
        data[i] = data[i] * 2
    return data

original = [1, 2, 3]
result = double_in_place(original)
print("original:", original)
print("result:  ", result)

# YOUR FIX HERE: rewrite double_in_place so `original` is left unchanged

### Sorted dictionary
Write a program which prints a sorted key list of a given dictionary. (_hint_: Check the `sorted` function, or try to sort the `keys()` function output)

In [ ]:
# YOUR CODE HERE
pass

### Copying a dictionary
Search how to copy a dictionary (different to making a reference)

In [ ]:
# YOUR CODE HERE
pass

### Polynomial
_Representing a polynomial by a dictionary_ : Consider the polynomial $p(x) = -1 + x^2 + 3x^7$ . It can be represent by means of a dictionary as `p = {0:-1, 2:1, 7:3}` (What is the advantage over using a list?). Write a function which gets a dictionary representing a polynomial, an `x` value, and returns the polynomial evaluated on that `x` value. 

In [ ]:
# YOUR CODE HERE
pass

### Histogram

Make a function which, given a dictionary with `values` as ints (a simple version of a histogram), prints the histogram counters as `=`. For example (the keys can be arbitrary)
   
    A : ========
    
    B : =============
    
    C : =======
    
    D : ==
    
    E : ==

In [ ]:
mydict = {'A':8, 'B':12, 'C':5, 'D':2, 'E':2}

# YOUR CODE HERE
pass
print_histo(mydict)


### Histogram of a text
Make a program which reads a text and counts the numbers of ocurrences for each word.


In [ ]:
# YOUR CODE HERE
pass

**Exercise:** *Counter refactor.* You already wrote a program that counts word occurrences
in a text using a plain dictionary. Rewrite it using `collections.Counter`, then use
`.most_common(5)` to print the 5 most frequent words. Compare how many lines each version
took.

In [ ]:
from collections import Counter

text = """the quick brown fox jumps over the lazy dog
the dog barks at the fox"""

# YOUR CODE HERE
# counts = Counter(...)
# print(counts.most_common(5))

### Dictionaries and lists for polynomial
Modify the previous program for representing a polynomial with a dictionary to represent it with a list. Use both representations for the polynomyal $-\frac{1}{2} + 2x^{100}$. Print both representations and use both to evaluate that polynomial at $x = 1.05$ .

In [ ]:
# YOUR CODE HERE
pass

### Polynomial derivative
By using the dictionary plynomial representation of the previous exercises, write a function which computes the derivative of a given polynomial and returns a dictionary representation of the new polynomial representing the derivative. Test it. 

In [ ]:
# YOUR CODE HERE
pass